# 01 — Feature Extraction and Creation



## 1. Imports and Configuration


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
RANDOM_SEED = 42


## 2. Project Root and Path Configuration


In [2]:
def find_project_root() -> Path:
    cur = Path.cwd().resolve()
    for p in [cur] + list(cur.parents):
        if (p / "data").exists() and (p / "experiments").exists():
            return p
    raise RuntimeError("Could not locate project root.")

PROJECT_ROOT = find_project_root()
INPUT_CSV = PROJECT_ROOT / "data" / "raw" / "tabular" / "samples" / "charity-patient.csv"
OUT_DIR = PROJECT_ROOT / "data" / "interim" / "features"
OUT_DIR.mkdir(parents=True, exist_ok=True)


## 3. Load Revised Experimental Dataset


In [3]:
df = pd.read_csv(INPUT_CSV)
print(f"Loaded {len(df)} rows x {df.shape[1]} columns")


Loaded 1000 rows x 71 columns


## 4. Inspect Available Columns


In [4]:
REQUIRED_COLS = ["household_size", "monthly_family_income", "monthly_household_obligations_expenses", "total_amount", "eligibility"]
missing = [c for c in REQUIRED_COLS if c not in df.columns]
print("Missing required columns:", missing)


Missing required columns: []


## 5. Numeric Conversion Helper


In [5]:
def to_num(col):
    if col not in df.columns:
        return pd.Series(np.nan, index=df.index)
    return pd.to_numeric(df[col].astype(str).str.replace(",", "", regex=False), errors="coerce")

hh = to_num("household_size").clip(lower=1)
inc = to_num("monthly_family_income")
exp = to_num("monthly_household_obligations_expenses")
cost = to_num("total_amount")


## 6. Feature Engineering Calculations


In [6]:
df["eng_per_capita_income"] = inc / hh
df["eng_expense_to_income_ratio"] = exp / inc.replace(0, np.nan)
df["eng_treatment_to_income_ratio"] = cost / inc.replace(0, np.nan)


## 7. Feature Validation & Missingness Check


In [7]:
eng_cols = ["eng_per_capita_income", "eng_expense_to_income_ratio", "eng_treatment_to_income_ratio"]
print(df[eng_cols].isna().sum())


eng_per_capita_income            155
eng_expense_to_income_ratio      257
eng_treatment_to_income_ratio    176
dtype: int64


## 8. Save Engineered Dataset


In [8]:
out_path = OUT_DIR / "feature_engineered_dataset.csv"
df.to_csv(out_path, index=False)
print("Saved:", out_path)


Saved: C:\Users\ASUS TUF\Desktop\Project in Program\Thesis Project\ofphil-classification-v2\data\interim\features\feature_engineered_dataset.csv


### Engineered Intake Ratios & Policy Alignment
Compute domain-safe intake ratios from approved predictors in `models/phase_05/feature_exclusion_policy.json` (excluding PII, `data_source`, and post-decision fields).

In [9]:
import json
import numpy as np
import pandas as pd

POLICY_PATH = PROJECT_ROOT / "models" / "phase_05" / "feature_exclusion_policy.json"
with open(POLICY_PATH, "r", encoding="utf-8") as f:
    policy = json.load(f)

df_raw = pd.read_csv(PROJECT_ROOT / "data" / "raw" / "tabular" / "samples" / "charity-patient.csv")
df_eng = df_raw[df_raw["eligibility"].isin(["ELIGIBLE", "INELIGIBLE"])].copy()

inc = pd.to_numeric(df_eng.get("monthly_family_income"), errors="coerce").fillna(0.0)
obl = pd.to_numeric(df_eng.get("monthly_household_obligations_expenses"), errors="coerce").fillna(0.0)
cost = pd.to_numeric(df_eng.get("cost_of_treatment"), errors="coerce").fillna(0.0)
dep = pd.to_numeric(df_eng.get("number_of_dependents"), errors="coerce").fillna(0.0)
hs = pd.to_numeric(df_eng.get("household_size"), errors="coerce").replace(0, np.nan)

df_eng["income_to_obligations_ratio"] = np.where(obl > 0, inc / obl, 0.0)
df_eng["cost_to_income_ratio"] = np.where(inc > 0, cost / inc, 0.0)
df_eng["dependents_per_household"] = (dep / hs).fillna(0.0)

print(f"Engineered ratios on {len(df_eng)} adjudicated rows. Approved base features: {len(policy['approved_features'])}")

Engineered ratios on 990 adjudicated rows. Approved base features: 27
